# CNN Development on Custom Dataset — Fresh vs Rotten Fruits Classification
**Course:** CVPR  |  **Student ID:** `YOUR_STUDENT_ID`  |  **Framework:** PyTorch

**Objective:** Design and train a custom CNN that classifies fruit images into 6 classes (fresh / rotten × apple / banana / orange).

**Dataset:** [Fruits fresh and rotten for classification (Kaggle)](https://www.kaggle.com/datasets/sriramr/fruits-fresh-and-rotten-for-classification)

**Notebook structure**
1. Import Libraries
2. Load and Explore Dataset
3. Data Preprocessing & Augmentation
4. Define CNN Architecture
5. Training Loop with Validation
6. Evaluate Model on Test Set
7. Visualizations (Loss/Accuracy Curves)
8. Save and Load Model
9. Analysis & Discussion of Results
10. Conclusions & Future Work

## 1. Import Libraries
All libraries used in the notebook are imported here. The `device` line selects the GPU when available (in Colab: *Runtime → Change runtime type → T4 GPU*) so training is much faster.

In [ ]:
# ---------------------------------------------------------------
# SECTION 1: IMPORT LIBRARIES
# ---------------------------------------------------------------
from torchvision import datasets
import torchvision.transforms as transforms

from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn

import os
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Use GPU if available, otherwise CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

## 2. Load and Explore Dataset
**Dataset description**
- Source: Kaggle (`sriramr/fruits-fresh-and-rotten-for-classification`)
- Content: RGB photos of apples, bananas and oranges, each labelled *fresh* or *rotten*
- Classes (6): `freshapples`, `freshbanana`, `freshoranges`, `rottenapples`, `rottenbanana`, `rottenoranges`
- Folder layout: `dataset/train/<class>/` and `dataset/test/<class>/`

The dataset is downloaded with `kagglehub` (no API key needed for public datasets).

In [ ]:
# ---------------------------------------------------------------
# SECTION 2: LOAD DATASET (download from Kaggle using kagglehub)
# ---------------------------------------------------------------
import kagglehub
path = kagglehub.dataset_download("sriramr/fruits-fresh-and-rotten-for-classification")
print("Path to dataset files:", path)

# Find the 'train' and 'test' folders automatically (folder nesting can differ between versions)
for root, dirs, files in os.walk(path):
    if 'train' in dirs and 'test' in dirs:
        dataset_root = root
        break
print("Dataset root:", dataset_root)

## 3. Data Preprocessing & Augmentation
- **Image size = 32 × 32** (all images are resized to the same size so they can be batched).
- **Training transform (with augmentation):** horizontal flip, rotation ±15°, and a *mild* colour jitter. Augmentation acts as a regulariser — it creates new variations every epoch and reduces over-fitting. Colour jitter is kept small because colour/darkness is the main cue that separates fresh from rotten fruit.
- **Validation / Test transform:** only resize + `ToTensor` (no random changes, so evaluation is deterministic).
- **Split:** the dataset already has `train` and `test` folders. As in the example notebook, the original test folder is split 50 % / 50 % into **validation** and **test** sets (fixed random seed for reproducibility).
- **Batch size = 32.**

In [ ]:
# ---------------------------------------------------------------
# SECTION 3: PREPROCESSING & AUGMENTATION
# ---------------------------------------------------------------
import torchvision.transforms as transforms
from torch.utils.data import random_split

size = 32   # input image size (32 x 32)

# Training transform: resize + augmentation
train_transform = transforms.Compose([
    transforms.Resize((size, size)),        # resize to model input size
    transforms.RandomHorizontalFlip(),      # horizontal flip augmentation
    transforms.RandomRotation(15),          # rotate +-15 degrees
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),  # mild colour change
    transforms.ToTensor()                   # convert PIL -> float tensor [0.0,1.0]
])

# Validation / test transform: NO augmentation
val_transform = transforms.Compose([
    transforms.Resize((size, size)),
    transforms.ToTensor()
])

train_folder = dataset_root + '/train'
test_folder = dataset_root + '/test'

# ImageFolder: each sub-folder name becomes a class label
train_data = datasets.ImageFolder(train_folder, transform = train_transform)
val_test_data = datasets.ImageFolder(test_folder, transform = val_transform)

# Split original test dataset into validation (50%) and test (50%)
val_size = len(val_test_data) // 2
test_size = len(val_test_data) - val_size

val_data, test_data = random_split(
    val_test_data,
    [val_size, test_size],
    generator = torch.Generator().manual_seed(42)   # fixed seed -> same split every run
)

In [ ]:
# Dataset summary
print('Dataset Summary')
print('Train Data: ', len(train_data))
print('Validation Data: ', len(val_data))
print('Test Data: ', len(test_data))
print('Classess and Indices: ' , train_data.class_to_idx)
num_classes = len(train_data.classes)   # number of output classes (6)
print('Number of classes: ', num_classes)

In [ ]:
# Class distribution of the training set (check for class imbalance)
counts = np.bincount(train_data.targets, minlength=num_classes)
plt.figure(figsize=(8, 3))
plt.bar(train_data.classes, counts)
plt.title("Training images per class")
plt.xticks(rotation=30)
plt.ylabel("Number of images")
plt.tight_layout()
plt.show()
print(dict(zip(train_data.classes, counts)))

In [ ]:
# Show a few random sample images (after resizing to 32x32) with their labels
ROWS, COLS = 2, 8
fig, axes = plt.subplots(ROWS, COLS, figsize=(14, 4))
for ax in axes.flat:
    img, label = val_test_data[np.random.randint(0, len(val_test_data))]
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(val_test_data.classes[label], fontsize=7)
    ax.set(xticks=[], yticks=[])
plt.tight_layout()
plt.show()

### Hyperparameters and rationale
| Hyperparameter | Value | Rationale |
|---|---|---|
| Image size | 32 × 32 | Required input size; small images keep training fast |
| Batch size | 32 | Required; good balance of gradient noise (helps generalisation) and speed |
| Hidden layer (fully connected) | 256 | Required; enough capacity to combine the conv features |
| Optimizer | Adam | Adaptive learning rate, converges fast and reliably on image data |
| Learning rate | 1e-3 | Standard starting value for Adam; reduced automatically by the scheduler when validation loss stops improving |
| Weight decay | 1e-4 | L2 regularisation to limit over-fitting |
| Dropout | 0.25 (conv) / 0.5 (FC) | Randomly drops units so the network does not memorise the training set |
| Epochs | 30 | Enough for convergence; the best epoch (highest validation accuracy) is kept |
| Loss | Cross-Entropy | Standard loss for multi-class classification |

In [ ]:
# ---------------------------------------------------------------
# HYPERPARAMETERS
# ---------------------------------------------------------------
batch_size = 32          # number of images per batch
learning_rate = 1e-3     # initial learning rate for Adam
epoch_number = 30        # number of training epochs
hidden_size = 256        # neurons in the fully connected hidden layer
weight_decay = 1e-4      # L2 regularisation strength

In [ ]:
# ---------------------------------------------------------------
# DATA LOADERS (train is shuffled, validation/test are not)
# ---------------------------------------------------------------
trn_dl = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)
val_dl = DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=2)
test_dl = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=2)

# Check one batch shape: [batch, channels, height, width]
x_batch, y_batch = next(iter(trn_dl))
print("Batch of images:", x_batch.shape, "| Batch of labels:", y_batch.shape)

## 4. Define CNN Architecture
**Design (3 convolutional blocks + classifier):**

| Block | Layers | Output size |
|---|---|---|
| Input | RGB image | 3 × 32 × 32 |
| Block 1 | 2 × (Conv 3×3, 64 filters + BatchNorm + ReLU) → MaxPool | 64 × 16 × 16 |
| Block 2 | 2 × (Conv 3×3, 128 filters + BatchNorm + ReLU) → MaxPool → Dropout | 128 × 8 × 8 |
| Block 3 | Conv 3×3, 256 filters + BatchNorm + ReLU → MaxPool → Dropout | 256 × 4 × 4 |
| Classifier | Flatten → Linear(4096 → **256**) → ReLU → Dropout → Linear(256 → 6) | 6 logits |

**Justification**
- **Conv layers** learn spatial features: edges/texture in early layers (e.g. bruised spots, skin pattern), colour and shape combinations in deeper layers.
- **Filters double (64 → 128 → 256)** while the image size halves — deeper layers need more feature maps to describe more complex patterns.
- **`padding=1`** keeps the size unchanged inside a block, so no information at the borders is lost on a small 32×32 image.
- **BatchNorm** stabilises and speeds up training and allows a higher learning rate.
- **ReLU** is a simple non-linearity that avoids vanishing gradients.
- **MaxPool** halves the resolution, reduces computation and gives small translation invariance.
- **Dropout** reduces over-fitting.
- **Output layer has 6 neurons (raw logits)** — `nn.CrossEntropyLoss` applies softmax internally.

In [ ]:
# ---------------------------------------------------------------
# SECTION 4: CNN ARCHITECTURE
# get_model() returns the model, the loss function and the optimizer
# ---------------------------------------------------------------
from torch.optim import SGD, Adam
def get_model():
    model = nn.Sequential(
        # ---------- Feature Extraction ----------
        # Block 1: 3x32x32 -> 64x16x16
        nn.Conv2d(3, 64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.Conv2d(64, 64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2),

        # Block 2: 64x16x16 -> 128x8x8
        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.Conv2d(128, 128, kernel_size=3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Dropout(0.25),

        # Block 3: 128x8x8 -> 256x4x4
        nn.Conv2d(128, 256, kernel_size=3, padding=1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Dropout(0.25),

        # ---------- Classifier ----------
        nn.Flatten(),                         # 256*4*4 = 4096 features
        nn.Linear(256*4*4, hidden_size),      # hidden layer = 256
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(hidden_size, num_classes)   # 6 output classes (logits)
    ).to(device)

    # Cross-entropy loss for multi-class classification
    loss_fn = nn.CrossEntropyLoss()
    # Adam optimizer with L2 weight decay
    optimizer = Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    return model, loss_fn, optimizer

In [ ]:
# Detailed model summary (layers, output shapes, number of parameters)
!pip install -q torchsummary
from torchsummary import summary
model, loss_fn, optimizer = get_model()
summary(model, input_size=(3,32,32));

## 5. Training Loop with Validation
- `train_one_epoch`: one pass over the training data (forward → loss → backward → optimizer step).
- `validation`: evaluates on validation/test data with gradients disabled (`@torch.no_grad()`) and `model.eval()` (turns off Dropout and fixes BatchNorm).
- **Learning-rate scheduler (`ReduceLROnPlateau`)**: halves the learning rate when the validation loss does not improve for 3 epochs — big steps early, fine steps later for better final accuracy.
- **Best-model checkpoint:** the weights of the epoch with the highest validation accuracy are kept and used for the final test.

In [ ]:
# ---------------------------------------------------------------
# SECTION 5: TRAINING & VALIDATION FUNCTIONS
# ---------------------------------------------------------------
def train_one_epoch(data_loader, model, loss_fn, optimizer):
    model.train()                                  # enable dropout / batchnorm training mode
    epoch_losses, epoch_accuracies = [], []
    for x, y in data_loader:
        x, y = x.to(device), y.to(device)          # move batch to GPU/CPU

        optimizer.zero_grad()                      # clear old gradients
        prediction = model(x)                      # forward pass
        batch_loss = loss_fn(prediction, y)        # cross-entropy loss
        batch_loss.backward()                      # back-propagation
        optimizer.step()                           # update weights

        epoch_losses.append(batch_loss.item())
        is_correct = (prediction.argmax(-1) == y).tolist()   # correct predictions in batch
        epoch_accuracies.extend(is_correct)
    return np.mean(epoch_losses), np.mean(epoch_accuracies)

@torch.no_grad()
def validation(data_loader, model, loss_fn):
    model.eval()                                   # disable dropout, freeze batchnorm stats
    val_losses, val_accuracies = [], []
    for x, y in data_loader:
        x, y = x.to(device), y.to(device)          # move to device here
        prediction = model(x)
        val_losses.append(loss_fn(prediction, y).item())
        is_correct = (prediction.argmax(-1) == y).tolist()
        val_accuracies.extend(is_correct)
    return np.mean(val_losses), np.mean(val_accuracies)

In [ ]:
# ---------------------------------------------------------------
# TRAINING LOOP (30 epochs)
# ---------------------------------------------------------------
import copy

# Create a fresh model, loss function and optimizer
model, loss_fn, optimizer = get_model()

# Reduce learning rate by half if validation loss does not improve for 3 epochs
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

train_losses, train_accuracies = [], []
val_losses, val_accuracies = [], []

best_val_acc = 0.0                       # best validation accuracy seen so far
best_weights = copy.deepcopy(model.state_dict())

for epoch in range(epoch_number):
  epoch_loss, epoch_accuracy = train_one_epoch(trn_dl,model,loss_fn,optimizer)
  train_losses.append(epoch_loss)
  train_accuracies.append(epoch_accuracy)

  v_loss, v_acc =  validation(val_dl,model,loss_fn)
  val_losses.append(v_loss)
  val_accuracies.append(v_acc)

  scheduler.step(v_loss)                 # adjust learning rate using validation loss

  # Keep the weights of the best epoch (highest validation accuracy)
  if v_acc > best_val_acc:
      best_val_acc = v_acc
      best_weights = copy.deepcopy(model.state_dict())

  print(f"Epoch [{epoch + 1}/{epoch_number}]"
        f" | Train Loss: {epoch_loss:.4f} | Train Accuracy: {epoch_accuracy*100:.2f}%"
        f" | Val Loss: {v_loss:.4f} | Val Accuracy: {v_acc*100:.2f}%"
        f" | LR: {optimizer.param_groups[0]['lr']:.6f}")

print(f"\nBest Validation Accuracy: {best_val_acc*100:.2f}%")

## 6. Evaluate Model on Test Set
The best checkpoint (highest validation accuracy) is loaded and evaluated **once** on the unseen test set. Besides accuracy we report precision, recall and F1-score per class and a confusion matrix, because accuracy alone can hide which classes are confused.

In [ ]:
# ---------------------------------------------------------------
# SECTION 6: TEST SET EVALUATION
# ---------------------------------------------------------------
model.load_state_dict(best_weights)      # load the best epoch's weights

test_loss, test_acc =  validation(test_dl,model,loss_fn)
print(f'Test Loss : {test_loss:0.4f} , Test Accuracy: {test_acc*100:0.2f}%')

In [ ]:
# Per-class metrics (precision, recall, F1) and confusion matrix
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

@torch.no_grad()
def get_predictions(data_loader, model):
    model.eval()
    all_preds, all_labels = [], []
    for x, y in data_loader:
        x = x.to(device)
        all_preds.extend(model(x).argmax(-1).cpu().tolist())   # predicted class
        all_labels.extend(y.tolist())                          # true class
    return all_labels, all_preds

y_true, y_pred = get_predictions(test_dl, model)
print(classification_report(y_true, y_pred, target_names=train_data.classes, digits=4))

fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=train_data.classes).plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title("Confusion Matrix (Test Set)")
plt.tight_layout()
plt.show()

In [ ]:
# Predict a single test image with confidence (softmax probability)
@torch.no_grad()
def predictOne(model, img, classes = None):
  model.eval()
  img = img.view(-1,3,size,size).to(device)
  output = model(img)
  pred = torch.exp(output)/torch.sum(torch.exp(output))     # softmax
  confidence, index = pred.max(-1)
  plt.figure(figsize=(2,2))
  plt.imshow(img.view(3,size,size).permute(1,2,0).cpu().numpy())
  plt.title(f' {classes[index]} , Confidence: {confidence.item()*100:0.2f}%', fontsize=8)
  plt.axis('off')
  plt.show()

test_img, label = test_data[2]
print("True label:", train_data.classes[label])
predictOne(model, test_img, train_data.classes)

## 7. Visualizations (Loss/Accuracy Curves)
Training and validation loss/accuracy are plotted per epoch. They show how fast the model learns, whether it over-fits (training keeps improving while validation stalls or worsens) and whether training has converged.

In [ ]:
# ---------------------------------------------------------------
# SECTION 7: LOSS & ACCURACY CURVES
# ---------------------------------------------------------------
epochs = np.arange(epoch_number) + 1
plt.figure(figsize=(15, 4))
plt.subplot(121)
plt.title("Loss value over increasing epochs")
plt.plot(epochs, train_losses, marker='o', label="Training Loss")
plt.plot(epochs, val_losses, marker='o', label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.xticks(epochs[::2])
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(122)
plt.title("Accuracy over Epochs")
plt.plot(epochs, train_accuracies, marker='o', label="Training Accuracy")
plt.plot(epochs, val_accuracies, marker='o', label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.xticks(epochs[::2])
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 8. Save and Load Model
Only the weights (`state_dict`) are saved to a `.pth` file. To reuse the model, create the same architecture with `get_model()` and load the weights. The loaded model must give the same test accuracy as the original (sanity check). Per the assignment: if the file is larger than 20 MB, keep it locally and do not upload it to GitHub.

In [ ]:
# ---------------------------------------------------------------
# SECTION 8: SAVE THE TRAINED MODEL WEIGHTS (.pth)
# ---------------------------------------------------------------
torch.save(model.state_dict(),'model.pth')
print("Model size: %.2f MB" % (os.path.getsize('model.pth')/1024/1024))

# Optional (Colab): download the file to your computer
# from google.colab import files
# files.download('model.pth')

In [ ]:
# Load the saved weights into a new model and verify the test accuracy is the same
saved_model, saved_loss_fn, _ = get_model()
saved_model.load_state_dict(torch.load('model.pth', map_location=device))
saved_model.eval()
summary(saved_model,(3,32,32))

t_loss, t_acc = validation(test_dl, saved_model, saved_loss_fn)
print(f'Loaded Model -> Test Loss : {t_loss:0.4f} , Test Accuracy: {t_acc*100:0.2f}%')

## 9. Analysis & Discussion of Results
> *Fill in with your own numbers after running the notebook (the viva will ask for justification).*

- **Final results:** Best validation accuracy = `__ %`, Test accuracy = `__ %`, Test loss = `__`.
- **Learning curves:** Describe whether the training and validation curves are close (good generalisation) or whether a gap appears (over-fitting). Mention the epoch where the validation accuracy peaked and where the learning rate was reduced.
- **Confusion matrix:** Name the most confused classes (e.g. `freshapples` vs `rottenapples`) and give a reason — at 32×32 resolution small bruises or spots can be lost.
- **Effect of design choices:**
  - BatchNorm + Adam → fast and stable convergence.
  - Dropout + augmentation + weight decay → smaller gap between training and validation accuracy.
  - LR scheduler + best-checkpoint saving → better final accuracy than training with a fixed learning rate and taking the last epoch.

## 10. Conclusions & Future Work
**Conclusion:** A custom 5-convolution-layer CNN with BatchNorm, Dropout and a 256-unit hidden layer was trained on the fresh/rotten fruit dataset using Adam and cross-entropy loss for 30 epochs, batch size 32, image size 32×32. The model reached `__ %` test accuracy. The weights were saved to `model.pth` and reloaded successfully.

**Limitations:** 32×32 inputs lose fine detail (small spots/mould); only three fruit types; backgrounds in the dataset are fairly clean.

**Future work:**
- Use higher resolution (e.g. 64×64 or 128×128) to preserve texture detail.
- Try transfer learning with a pre-trained network (ResNet18 / MobileNet) fine-tuned on this dataset.
- Add stronger augmentation and test-time augmentation.
- Add more fruit types and real-world (cluttered background) images, and deploy as a mobile/web app for food-quality checking.